In [17]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image_path = "images\\truck.jpg"

# Constants
DISPLAY_WIDTH = 1024
DISPLAY_HEIGHT = 768

In [18]:
# Global variables
input_points = []
input_labels = []
input_boxes = []
box_labels = []
mode = None  # "point" or "box"
box_clicks = []
scale_x = 1.0
scale_y = 1.0

def select_mode():
    global mode
    print("Select mode:")
    print("1 - Select multiple points (label 1 = left click, 0 = right click)")
    print("2 - Select multiple bounding boxes (label 1 = right click first, 0 = left click first)")
    while True:
        choice = input("Enter 1 or 2: ")
        if choice == "1":
            mode = "point"
            break
        elif choice == "2":
            mode = "box"
            break
        else:
            print("Invalid choice. Try again.")

def click_event(event, x, y, flags, param):
    global input_points, input_labels, input_boxes, box_labels, box_clicks

    # Scale coordinates back to original image size
    original_x = int(x / scale_x)
    original_y = int(y / scale_y)

    if mode == "point":
        if event == cv2.EVENT_LBUTTONDOWN:
            input_points.append([original_x, original_y])
            input_labels.append(1)
            print(f"Added point (label=1): ({original_x}, {original_y})")
            cv2.circle(display_img, (x, y), 5, (0, 255, 0), -1)
            cv2.imshow("Image", display_img)

        elif event == cv2.EVENT_RBUTTONDOWN:
            input_points.append([original_x, original_y])
            input_labels.append(0)
            print(f"Added point (label=0): ({original_x}, {original_y})")
            cv2.circle(display_img, (x, y), 5, (0, 0, 255), -1)
            cv2.imshow("Image", display_img)

    elif mode == "box":
        if event == cv2.EVENT_LBUTTONDOWN or event == cv2.EVENT_RBUTTONDOWN:
            label = 1 if event == cv2.EVENT_RBUTTONDOWN else 0
            box_clicks.append((x, y, label))

            if len(box_clicks) == 2:
                (x1, y1, label1), (x2, y2, _) = box_clicks
                orig_x1 = int(min(x1, x2) / scale_x)
                orig_y1 = int(min(y1, y2) / scale_y)
                orig_x2 = int(max(x1, x2) / scale_x)
                orig_y2 = int(max(y1, y2) / scale_y)
                input_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2])
                box_labels.append(label1)
                print(f"Added box (label={label1}): {[orig_x1, orig_y1, orig_x2, orig_y2]}")

                color = (255, 0, 0) if label1 == 0 else (0, 255, 255)
                cv2.rectangle(display_img, (min(x1, x2), min(y1, y2)), (max(x1, x2), max(y1, y2)), color, 2)
                cv2.imshow("Image", display_img)
                box_clicks = []

def main(image_path):
    global display_img, scale_x, scale_y
    select_mode()

    original_img = cv2.imread(image_path)
    if original_img is None:
        print("Could not load image.")
        return

    # Resize image for display
    h, w = original_img.shape[:2]
    scale = min(DISPLAY_WIDTH / w, DISPLAY_HEIGHT / h)
    new_size = (int(w * scale), int(h * scale))
    display_img = cv2.resize(original_img, new_size)

    scale_x = scale
    scale_y = scale

    cv2.imshow("Image", display_img)
    cv2.setMouseCallback("Image", click_event)
    print("Click on the image. Press any key to finish.")

    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # Output results
    if mode == "point" and input_points:
        input_point = np.array(input_points)
        input_label = np.array(input_labels)
        print("\nSaved points as:")
        print(f"input_point = np.array({input_point.tolist()})")
        print(f"input_label = np.array({input_label.tolist()})")

    elif mode == "box" and input_boxes:
        input_box = np.array(input_boxes)
        input_label = np.array(box_labels)
        print("\nSaved boxes as:")
        print(f"input_box = np.array({input_box.tolist()})")
        print(f"input_label = np.array({input_label.tolist()})")

    else:
        print("No selection made.")

main(image_path)

Select mode:
1 - Select multiple points (label 1 = left click, 0 = right click)
2 - Select multiple bounding boxes (label 1 = right click first, 0 = left click first)
Click on the image. Press any key to finish.
Added point (label=1): (212, 298)
Added point (label=1): (481, 493)
Added point (label=0): (785, 444)
Added point (label=0): (627, 335)
Added point (label=0): (1128, 543)
Added point (label=1): (878, 643)
Added point (label=1): (1024, 370)
Added point (label=0): (1406, 504)
Added point (label=1): (1204, 785)
Added point (label=0): (560, 789)

Saved points as:
input_point = np.array([[212, 298], [481, 493], [785, 444], [627, 335], [1128, 543], [878, 643], [1024, 370], [1406, 504], [1204, 785], [560, 789]])
input_label = np.array([1, 1, 0, 0, 0, 1, 1, 0, 1, 0])
